
# Notebook 16 — Critical Scaling and Universality Windows

Notebook 15 showed:

```text
renormalized bounded transition structure persists across finite graph size
```

Notebook 16 asks:

> where does universality persist, and where does topology-dependent fragmentation emerge?

This notebook replaces saturation-prone hard-threshold extraction with:

- midpoint transition extraction,
- derivative-based transition analysis,
- finite universality-window detection,
- critical scaling fits.


## Imports and setup

In [ ]:

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from scipy.optimize import curve_fit
from scipy.interpolate import interp1d

np.random.seed(42)

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

FIG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

PHASE_LOCK_THRESHOLD = 24 / 25
MIDPOINT_LEVEL = 0.50
EPSILON = 0.05

GRAPH_SIZES = [16, 32, 64, 128]
NOISE_GRID = np.linspace(0.0, 0.40, 81)

print("Ready.")
print(f"phase-lock threshold = {PHASE_LOCK_THRESHOLD:.3f}")
print(f"midpoint level = {MIDPOINT_LEVEL:.2f}")
print(f"universality epsilon = {EPSILON:.2f}")



## Shared bounded transition model

We use the shared logistic profile:

```text
f(z) = 1 / (1 + exp(-z))
```

with collapse variable:

```text
z = (η - η_c) / σ
```

where:

- `η` = link noise,
- `η_c` = transition midpoint,
- `σ` = transition width.


In [ ]:

def logistic_z(z):
    return 1 / (1 + np.exp(-z))

def logistic_noise(eta, eta_c, sigma):
    sigma = max(abs(float(sigma)), 1e-6)
    return 1 / (1 + np.exp(-(eta - eta_c) / sigma))

def finite_size_form(N, eta_inf, a, nu):
    return eta_inf + a * (N ** (-nu))

def power_law_width(N, b, beta, floor):
    return floor + b * (N ** (-beta))


## Topology parameters

In [ ]:

TOPOLOGY_PARAMS = {
    "ring_lattice": {
        "eta_inf": 0.135,
        "eta_shift": 0.060,
        "sigma_inf": 0.030,
        "sigma_scale": 0.080,
        "nu": 0.45,
        "beta": 0.40,
        "fragment_strength": 0.08,
        "modifier": 0.98,
    },
    "small_world": {
        "eta_inf": 0.160,
        "eta_shift": 0.075,
        "sigma_inf": 0.038,
        "sigma_scale": 0.095,
        "nu": 0.48,
        "beta": 0.37,
        "fragment_strength": 0.06,
        "modifier": 1.02,
    },
    "erdos_renyi": {
        "eta_inf": 0.120,
        "eta_shift": 0.055,
        "sigma_inf": 0.028,
        "sigma_scale": 0.075,
        "nu": 0.42,
        "beta": 0.45,
        "fragment_strength": 0.10,
        "modifier": 0.96,
    },
    "scale_free": {
        "eta_inf": 0.105,
        "eta_shift": 0.052,
        "sigma_inf": 0.025,
        "sigma_scale": 0.070,
        "nu": 0.44,
        "beta": 0.50,
        "fragment_strength": 0.14,
        "modifier": 0.93,
    },
    "modular_clustered": {
        "eta_inf": 0.092,
        "eta_shift": 0.048,
        "sigma_inf": 0.023,
        "sigma_scale": 0.065,
        "nu": 0.40,
        "beta": 0.52,
        "fragment_strength": 0.18,
        "modifier": 0.90,
    },
}

TOPOLOGIES = list(TOPOLOGY_PARAMS.keys())

params_df = pd.DataFrame([{"topology": k, **v} for k, v in TOPOLOGY_PARAMS.items()])
params_df


## Graph generators and diagnostics

In [ ]:

def make_ring_lattice(N, k=4):
    k = min(k, N - 1)
    if k % 2 == 1:
        k -= 1
    return nx.watts_strogatz_graph(N, k, 0.0, seed=42)

def make_small_world(N, k=4, p=0.2):
    k = min(k, N - 1)
    if k % 2 == 1:
        k -= 1
    return nx.watts_strogatz_graph(N, k, p, seed=42)

def make_erdos_renyi(N, p=None):
    if p is None:
        p = min(0.35, max(0.08, 4.0 / N))
    G = nx.erdos_renyi_graph(N, p, seed=42)
    if not nx.is_connected(G):
        comps = list(nx.connected_components(G))
        for a, b in zip(comps[:-1], comps[1:]):
            G.add_edge(next(iter(a)), next(iter(b)))
    return G

def make_scale_free(N, m=2):
    m = min(m, max(1, N - 1))
    return nx.barabasi_albert_graph(N, m, seed=42)

def make_modular_clustered(N, blocks=4, p_in=0.28, p_out=0.025):
    blocks = min(blocks, N)
    base = N // blocks
    sizes = [base] * blocks
    sizes[-1] += N - sum(sizes)

    probs = np.full((blocks, blocks), p_out)
    np.fill_diagonal(probs, p_in)

    G = nx.stochastic_block_model(sizes, probs, seed=42)
    if not nx.is_connected(G):
        comps = list(nx.connected_components(G))
        for a, b in zip(comps[:-1], comps[1:]):
            G.add_edge(next(iter(a)), next(iter(b)))
    return G

TOPOLOGY_BUILDERS = {
    "ring_lattice": make_ring_lattice,
    "small_world": make_small_world,
    "erdos_renyi": make_erdos_renyi,
    "scale_free": make_scale_free,
    "modular_clustered": make_modular_clustered,
}

diag_rows = []

for N in GRAPH_SIZES:
    for topology, builder in TOPOLOGY_BUILDERS.items():
        G = builder(N)

        degrees = np.array([d for _, d in G.degree()], dtype=float)

        if nx.is_connected(G):
            path_length = nx.average_shortest_path_length(G)
        else:
            largest = max(nx.connected_components(G), key=len)
            path_length = nx.average_shortest_path_length(G.subgraph(largest))

        diag_rows.append({
            "n_modules": N,
            "topology": topology,
            "average_degree": float(np.mean(degrees)),
            "degree_variance": float(np.var(degrees)),
            "clustering": float(nx.average_clustering(G)),
            "path_length": float(path_length),
            "largest_component_fraction": float(len(max(nx.connected_components(G), key=len)) / N),
        })

diag_df = pd.DataFrame(diag_rows)

diag_path = RESULTS_DIR / "critical_scaling_graph_diagnostics.csv"
diag_df.to_csv(diag_path, index=False)

print(f"saved: {diag_path}")
diag_df.head()



## Projection-response simulation

Instead of thresholding projection success directly, Notebook 16 simulates a bounded coherence curve over noise.

The model creates a topology-specific transition midpoint and width:

```text
η_c(N) = η_inf + a N^(-ν)
σ(N) = σ_inf + b N^(-β)
```

Then it adds bounded topology-specific deviations outside a central universality window.


In [ ]:

def topology_eta_c(topology, N):
    p = TOPOLOGY_PARAMS[topology]
    return p["eta_inf"] + p["eta_shift"] * (N ** (-p["nu"]))

def topology_sigma(topology, N):
    p = TOPOLOGY_PARAMS[topology]
    return p["sigma_inf"] + p["sigma_scale"] * (N ** (-p["beta"]))

def simulate_cgcs_curve(topology, N, noise_grid, repeat=0):
    p = TOPOLOGY_PARAMS[topology]

    eta_c = topology_eta_c(topology, N)
    sigma = topology_sigma(topology, N)

    # Coherence decreases as noise increases.
    z = (noise_grid - eta_c) / sigma
    base = 1 - logistic_z(z)

    # Fragmentation emerges mostly outside central window and is topology-specific.
    central_weight = np.exp(-0.5 * z**2)
    outside_weight = 1 - central_weight

    fragment = p["fragment_strength"] * outside_weight * logistic_z((noise_grid - eta_c) / (2.0 * sigma))

    # Finite-size stochastic perturbation, reduced with N.
    rng = np.random.default_rng(
        40_000 + repeat + N + sum(ord(c) for c in topology)
    )
    noise_term = rng.normal(0, 0.010 * np.sqrt(32 / N), size=len(noise_grid))

    cgcs = p["modifier"] * base - fragment + noise_term
    cgcs = np.clip(cgcs, 0, 1)

    return cgcs


## Generate transition curves

In [ ]:

records = []
repeats = 24

for N in GRAPH_SIZES:
    for topology in TOPOLOGIES:
        all_curves = []

        for repeat in range(repeats):
            cgcs = simulate_cgcs_curve(topology, N, NOISE_GRID, repeat=repeat)
            all_curves.append(cgcs)

        all_curves = np.array(all_curves)
        mean_curve = all_curves.mean(axis=0)
        std_curve = all_curves.std(axis=0)

        for eta, mean_val, std_val in zip(NOISE_GRID, mean_curve, std_curve):
            records.append({
                "n_modules": N,
                "topology": topology,
                "link_noise": float(eta),
                "cgcs": float(mean_val),
                "cgcs_std": float(std_val),
            })

curve_df = pd.DataFrame(records)

curve_path = RESULTS_DIR / "critical_scaling_transition_curves.csv"
curve_df.to_csv(curve_path, index=False)

print(f"saved: {curve_path}")
curve_df.head()


## Midpoint and derivative extraction

In [ ]:

def extract_transition_metrics(noise, cgcs):
    noise = np.asarray(noise, dtype=float)
    cgcs = np.asarray(cgcs, dtype=float)

    # Sort defensively.
    order = np.argsort(noise)
    noise = noise[order]
    cgcs = cgcs[order]

    # Since CGCS decreases with noise, midpoint is crossing near 0.5.
    midpoint_idx = int(np.argmin(np.abs(cgcs - MIDPOINT_LEVEL)))
    eta_mid = float(noise[midpoint_idx])

    # Numerical derivative.
    deriv = np.gradient(cgcs, noise)
    max_slope_idx = int(np.argmax(np.abs(deriv)))
    eta_slope = float(noise[max_slope_idx])
    max_abs_slope = float(np.abs(deriv[max_slope_idx]))

    # Logistic slope at midpoint for decreasing profile: |slope|max ≈ 1/(4 sigma)
    sigma_est = float(1 / max(4 * max_abs_slope, 1e-6))

    return {
        "eta_midpoint": eta_mid,
        "eta_max_slope": eta_slope,
        "max_abs_slope": max_abs_slope,
        "sigma_derivative": sigma_est,
    }

metric_rows = []
deriv_rows = []

for N in GRAPH_SIZES:
    for topology in TOPOLOGIES:
        sub = curve_df[
            (curve_df["n_modules"] == N)
            & (curve_df["topology"] == topology)
        ].sort_values("link_noise")

        noise = sub["link_noise"].to_numpy()
        cgcs = sub["cgcs"].to_numpy()

        metrics = extract_transition_metrics(noise, cgcs)

        metric_rows.append({
            "n_modules": N,
            "topology": topology,
            **metrics,
        })

        deriv = np.gradient(cgcs, noise)

        for eta, dval in zip(noise, deriv):
            deriv_rows.append({
                "n_modules": N,
                "topology": topology,
                "link_noise": float(eta),
                "d_cgcs_d_noise": float(dval),
                "abs_slope": float(abs(dval)),
            })

metrics_df = pd.DataFrame(metric_rows)
deriv_df = pd.DataFrame(deriv_rows)

metrics_path = RESULTS_DIR / "critical_midpoint_scaling.csv"
deriv_path = RESULTS_DIR / "transition_derivative_profiles.csv"

metrics_df.to_csv(metrics_path, index=False)
deriv_df.to_csv(deriv_path, index=False)

print(f"saved: {metrics_path}")
print(f"saved: {deriv_path}")
metrics_df.head(10)


## Transition derivative profiles

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.ravel()

for ax, topology in zip(axes, TOPOLOGIES):
    for N in GRAPH_SIZES:
        sub = deriv_df[
            (deriv_df["topology"] == topology)
            & (deriv_df["n_modules"] == N)
        ]

        ax.plot(
            sub["link_noise"],
            sub["abs_slope"],
            linewidth=1.8,
            label=f"N={N}"
        )

    ax.set_title(topology.replace("_", " "))
    ax.set_xlabel("link noise")
    ax.set_ylabel("|d(CGCS)/dη|")
    ax.grid(alpha=0.3)

axes[-1].axis("off")
axes[0].legend(fontsize=8)

plt.tight_layout()

fig_path = FIG_DIR / "transition_derivative_profiles.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Midpoint scaling vs graph size

In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = metrics_df[metrics_df["topology"] == topology]
    plt.plot(
        sub["n_modules"],
        sub["eta_midpoint"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.xlabel("graph size N")
plt.ylabel("midpoint noise η_mid")
plt.title("Midpoint scaling vs graph size")
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "midpoint_scaling_vs_graph_size.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Renormalized collapse data

In [ ]:

collapse_rows = []

for N in GRAPH_SIZES:
    for topology in TOPOLOGIES:
        metric = metrics_df[
            (metrics_df["n_modules"] == N)
            & (metrics_df["topology"] == topology)
        ].iloc[0]

        eta_mid = float(metric["eta_midpoint"])
        sigma = max(float(metric["sigma_derivative"]), 1e-6)

        sub = curve_df[
            (curve_df["n_modules"] == N)
            & (curve_df["topology"] == topology)
        ].sort_values("link_noise")

        for _, row in sub.iterrows():
            z = (float(row["link_noise"]) - eta_mid) / sigma

            # Shared decreasing profile for CGCS.
            shared = 1 - logistic_z(z)
            residual = float(row["cgcs"]) - shared

            collapse_rows.append({
                "n_modules": N,
                "topology": topology,
                "link_noise": float(row["link_noise"]),
                "cgcs": float(row["cgcs"]),
                "z": float(z),
                "shared_profile": float(shared),
                "collapse_residual": float(residual),
                "abs_residual": float(abs(residual)),
            })

collapse_df = pd.DataFrame(collapse_rows)

collapse_path = RESULTS_DIR / "renormalized_collapse_data.csv"
collapse_df.to_csv(collapse_path, index=False)

print(f"saved: {collapse_path}")
collapse_df.head()


## Universality window detection

In [ ]:

window_rows = []

for N in GRAPH_SIZES:
    for topology in TOPOLOGIES:
        sub = collapse_df[
            (collapse_df["n_modules"] == N)
            & (collapse_df["topology"] == topology)
        ].sort_values("z")

        in_window = sub[sub["abs_residual"] <= EPSILON]

        if len(in_window) == 0:
            z_lower = np.nan
            z_upper = np.nan
            bandwidth = 0.0
            fraction = 0.0
        else:
            z_lower = float(in_window["z"].min())
            z_upper = float(in_window["z"].max())
            bandwidth = float(z_upper - z_lower)
            fraction = float(len(in_window) / len(sub))

        window_rows.append({
            "n_modules": N,
            "topology": topology,
            "z_lower": z_lower,
            "z_upper": z_upper,
            "universality_bandwidth": bandwidth,
            "universality_fraction": fraction,
            "mean_abs_residual": float(sub["abs_residual"].mean()),
        })

window_df = pd.DataFrame(window_rows)

window_path = RESULTS_DIR / "universality_window_summary.csv"
window_df.to_csv(window_path, index=False)

print(f"saved: {window_path}")
window_df.head(10)


## Universality window by topology

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.ravel()

for ax, topology in zip(axes, TOPOLOGIES):
    sub = window_df[window_df["topology"] == topology]

    ax.plot(
        sub["n_modules"],
        sub["universality_bandwidth"],
        marker="o",
        linewidth=2,
        label="bandwidth"
    )

    ax.set_title(topology.replace("_", " "))
    ax.set_xlabel("graph size N")
    ax.set_ylabel("universality bandwidth")
    ax.grid(alpha=0.3)

axes[-1].axis("off")

plt.tight_layout()

fig_path = FIG_DIR / "universality_window_by_topology.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Universality bandwidth vs graph size

In [ ]:

plt.figure(figsize=(10, 6))

for topology in TOPOLOGIES:
    sub = window_df[window_df["topology"] == topology]

    plt.plot(
        sub["n_modules"],
        sub["universality_bandwidth"],
        marker="o",
        linewidth=2,
        label=topology.replace("_", " ")
    )

plt.xlabel("graph size N")
plt.ylabel("universality bandwidth")
plt.title("Universality bandwidth vs graph size")
plt.grid(alpha=0.3)
plt.legend()

fig_path = FIG_DIR / "universality_bandwidth_vs_graph_size.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Renormalized universality collapse

In [ ]:

plt.figure(figsize=(10, 7))

for topology in TOPOLOGIES:
    sub = collapse_df[collapse_df["topology"] == topology]

    # plot a subset for readability
    plt.scatter(
        sub["z"],
        sub["cgcs"],
        s=25,
        alpha=0.35,
        label=topology.replace("_", " ")
    )

z_dense = np.linspace(-6, 6, 500)
plt.plot(
    z_dense,
    1 - logistic_z(z_dense),
    color="black",
    linewidth=4,
    linestyle="--",
    label="shared decreasing logistic"
)

plt.xlabel("renormalized collapse variable z")
plt.ylabel("CGCS")
plt.xlim(-6, 6)
plt.ylim(-0.02, 1.05)
plt.title("Renormalized universality collapse")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)

fig_path = FIG_DIR / "renormalized_universality_collapse.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Critical finite-size scaling fits

In [ ]:

scaling_rows = []

for topology in TOPOLOGIES:
    sub = metrics_df[metrics_df["topology"] == topology].copy()

    N = sub["n_modules"].to_numpy(dtype=float)
    eta = sub["eta_midpoint"].to_numpy(dtype=float)
    sigma = sub["sigma_derivative"].to_numpy(dtype=float)

    # Fit eta_c(N) = eta_inf + a N^(-nu)
    try:
        eta_params, _ = curve_fit(
            finite_size_form,
            N,
            eta,
            p0=[eta.min(), 0.05, 0.4],
            bounds=([0.0, -1.0, 0.01], [1.0, 1.0, 3.0]),
            maxfev=20000,
        )
    except Exception:
        eta_params = [np.nan, np.nan, np.nan]

    # Fit sigma(N) = floor + b N^(-beta)
    try:
        sig_params, _ = curve_fit(
            power_law_width,
            N,
            sigma,
            p0=[0.05, 0.4, sigma.min()],
            bounds=([0.0, 0.01, 0.0], [1.0, 3.0, 1.0]),
            maxfev=20000,
        )
    except Exception:
        sig_params = [np.nan, np.nan, np.nan]

    scaling_rows.append({
        "topology": topology,
        "eta_inf_est": float(eta_params[0]),
        "eta_amplitude_est": float(eta_params[1]),
        "nu_est": float(eta_params[2]),
        "sigma_amplitude_est": float(sig_params[0]),
        "beta_est": float(sig_params[1]),
        "sigma_floor_est": float(sig_params[2]),
    })

scaling_df = pd.DataFrame(scaling_rows)

scaling_path = RESULTS_DIR / "critical_scaling_fit_summary.csv"
scaling_df.to_csv(scaling_path, index=False)

print(f"saved: {scaling_path}")
scaling_df


## Critical midpoint scaling fits

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

N_dense = np.linspace(min(GRAPH_SIZES), max(GRAPH_SIZES), 300)

for topology in TOPOLOGIES:
    sub = metrics_df[metrics_df["topology"] == topology]
    fit = scaling_df[scaling_df["topology"] == topology].iloc[0]

    axes[0].scatter(
        sub["n_modules"],
        sub["eta_midpoint"],
        s=80,
        label=topology.replace("_", " ")
    )

    if not pd.isna(fit["eta_inf_est"]):
        axes[0].plot(
            N_dense,
            finite_size_form(
                N_dense,
                fit["eta_inf_est"],
                fit["eta_amplitude_est"],
                fit["nu_est"],
            ),
            linewidth=2
        )

    axes[1].scatter(
        sub["n_modules"],
        sub["sigma_derivative"],
        s=80,
        label=topology.replace("_", " ")
    )

    if not pd.isna(fit["beta_est"]):
        axes[1].plot(
            N_dense,
            power_law_width(
                N_dense,
                fit["sigma_amplitude_est"],
                fit["beta_est"],
                fit["sigma_floor_est"],
            ),
            linewidth=2
        )

axes[0].set_title("Midpoint scaling")
axes[0].set_xlabel("graph size N")
axes[0].set_ylabel("η_mid")

axes[1].set_title("Transition width scaling")
axes[1].set_xlabel("graph size N")
axes[1].set_ylabel("sigma estimate")

for ax in axes:
    ax.grid(alpha=0.3)

axes[1].legend(fontsize=8)

plt.tight_layout()

fig_path = FIG_DIR / "critical_scaling_fits.png"
plt.savefig(fig_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"saved: {fig_path}")


## Universality window summary table

In [ ]:

combined_df = (
    metrics_df
    .merge(window_df, on=["n_modules", "topology"], how="left")
    .merge(diag_df, on=["n_modules", "topology"], how="left")
)

combined_path = RESULTS_DIR / "universality_bandwidth_scaling.csv"
combined_df.to_csv(combined_path, index=False)

print(f"saved: {combined_path}")
combined_df.head(10)


## Summary export

In [ ]:

valid_windows = window_df.dropna(subset=["universality_bandwidth"])

if len(valid_windows) > 0:
    widest = valid_windows.sort_values(
        "universality_bandwidth",
        ascending=False
    ).iloc[0]

    widest_window = {
        "topology": widest["topology"],
        "n_modules": int(widest["n_modules"]),
        "universality_bandwidth": float(widest["universality_bandwidth"]),
        "universality_fraction": float(widest["universality_fraction"]),
    }

    mean_bandwidth = float(valid_windows["universality_bandwidth"].mean())
    mean_abs_residual = float(valid_windows["mean_abs_residual"].mean())
else:
    widest_window = None
    mean_bandwidth = None
    mean_abs_residual = None

summary = {
    "notebook": "16_critical_scaling_and_universality_window.ipynb",
    "phase_lock_threshold": PHASE_LOCK_THRESHOLD,
    "midpoint_level": MIDPOINT_LEVEL,
    "epsilon": EPSILON,
    "graph_sizes": GRAPH_SIZES,

    "core_claim": (
        "Topology-dependent finite systems exhibit bounded universality windows "
        "within which renormalized transition structure persists. Outside those "
        "windows, topology-specific fragmentation emerges."
    ),

    "interpretation": (
        "Midpoint and derivative-based extraction reduce saturation artifacts "
        "and expose finite universality windows across topology and graph size."
    ),

    "widest_window": widest_window,
    "mean_universality_bandwidth": mean_bandwidth,
    "mean_abs_residual": mean_abs_residual,

    "topologies": TOPOLOGIES,

    "figures": [
        "transition_derivative_profiles.png",
        "midpoint_scaling_vs_graph_size.png",
        "universality_window_by_topology.png",
        "universality_bandwidth_vs_graph_size.png",
        "renormalized_universality_collapse.png",
        "critical_scaling_fits.png",
    ],

    "results": [
        "critical_scaling_graph_diagnostics.csv",
        "critical_scaling_transition_curves.csv",
        "critical_midpoint_scaling.csv",
        "transition_derivative_profiles.csv",
        "renormalized_collapse_data.csv",
        "universality_window_summary.csv",
        "universality_bandwidth_scaling.csv",
        "critical_scaling_fit_summary.csv",
        "critical_scaling_summary.json",
    ],
}

summary_path = RESULTS_DIR / "critical_scaling_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

doc_lines = [
    "# Notebook 16 — Critical Scaling and Universality Windows",
    "",
    "**Core claim:** topology-dependent finite systems exhibit bounded universality windows within which renormalized transition structure persists.",
    "",
    "Main outputs:",
    "",
    "- `figures/transition_derivative_profiles.png`",
    "- `figures/midpoint_scaling_vs_graph_size.png`",
    "- `figures/universality_window_by_topology.png`",
    "- `figures/universality_bandwidth_vs_graph_size.png`",
    "- `figures/renormalized_universality_collapse.png`",
    "- `figures/critical_scaling_fits.png`",
    "",
]

doc_path = DOCS_DIR / "notebook_16_critical_scaling_and_universality_window.md"
doc_path.write_text("\n".join(doc_lines), encoding="utf-8")

print(json.dumps(summary, indent=2))
print(f"saved: {summary_path}")
print(f"saved: {doc_path}")



## Final interpretation

Notebook 16 moves from persistence heuristics toward universality-window extraction.

Careful conclusion:

```text
Topology-dependent finite systems exhibit bounded universality windows
within which renormalized transition structure persists.
Outside those windows, topology-specific fragmentation emerges.
```

This makes the sequence:

```text
thresholds → topology persistence → sharpness → finite-size scaling → universality windows
```


## Optional export zip

In [ ]:

zip_path = Path("notebook_16_outputs.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file():
                    zf.write(file)

print(f"Created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))
